# Experiment: Depth Pro GPU smoke test in Colab

**Question:** How much faster is one Colab GPU than the laptop CPU for the same Depth Pro checkpoint and three Replica frames?

This is a three-frame smoke test, not a full-sequence experiment. It uses float32 inference and the existing coursework scripts. It requires a Google account and an available Colab GPU. No UCL CS account is needed.

1. Upload this notebook through **File ? Upload notebook** in [Colab](https://colab.research.google.com/).
2. Before connecting, choose **Runtime ? Change runtime type ? GPU**, and set **Runtime version to 2026.07**. Connecting to a past runtime may take a few minutes.
3. Run cells in order and upload `cw1_gpu_smoke_inputs.zip` when prompted.
4. Download the results ZIP at the end. Keep the notebook private; no public repository is needed.

Free GPU availability is not guaranteed. The first cell stops before downloads if a GPU is unavailable. The model weights are about 1.9 GB and are downloaded directly into the temporary Colab VM.

Sources: [Colab runtime versions](https://research.google.com/colaboratory/runtime-version-faq.html), [resource limits](https://research.google.com/colaboratory/faq.html), [official Depth Pro](https://github.com/apple-aiml-research/ml-depth-pro).

**Validation:** notebook syntax and portable input extraction were checked locally. GPU execution, GPU memory requirements and the Colab upload/download dialogs remain to be tested in your session.

## 1. Check the runtime and GPU

Use runtime **2026.07** (Python 3.12). The current latest runtime uses a newer Python that conflicts with the original Depth Pro NumPy requirement. This check uses a fresh process; it does not import numerical libraries into the notebook kernel.

In [ ]:
import json
from pathlib import Path
import subprocess
import sys

if sys.version_info[:2] != (3, 12):
    raise RuntimeError("Select Runtime > Change runtime type > Runtime version 2026.07, then reconnect and rerun.")
probe_code = """import json, torch
available = torch.cuda.is_available()
print(json.dumps({'cuda_available': available, 'torch': torch.__version__, 'cuda_version': torch.version.cuda,
                  'gpu': torch.cuda.get_device_name(0) if available else None,
                  'gpu_memory_gib': torch.cuda.get_device_properties(0).total_memory / 2**30 if available else None}))
"""
gpu_info = json.loads(subprocess.check_output([sys.executable, "-c", probe_code], text=True))
if not gpu_info['cuda_available']:
    raise RuntimeError("No CUDA GPU is available. Select a GPU runtime; if Colab has no free allocation, stop here and try later.")
print(json.dumps(gpu_info, indent=2))
subprocess.run(["nvidia-smi"], check=True)

## 2. Upload the small test bundle

Select **cw1_gpu_smoke_inputs.zip**, not the full coursework repository or the model checkpoint. This contains three RGB/depth pairs, camera calibration, three pose rows and the scripts. The archive is extracted into temporary Colab storage.

In [ ]:
from google.colab import files
import io
import stat
import zipfile

uploaded = files.upload()
archives = [name for name in uploaded if name.endswith('.zip')]
if len(archives) != 1:
    raise ValueError("Upload exactly one ZIP: cw1_gpu_smoke_inputs.zip")
workspace = Path('/content/cw1_depthpro_test')
workspace.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(io.BytesIO(uploaded[archives[0]])) as archive:
    if sum(info.file_size for info in archive.infolist()) > 50 * 1024**2:
        raise ValueError("This smoke-test bundle should be below 50 MB")
    for info in archive.infolist():
        member = Path(info.filename)
        destination = (workspace / member).resolve()
        if (member.is_absolute() or '..' in member.parts or not member.parts
                or member.parts[0] != 'cw1_gpu_smoke'
                or not destination.is_relative_to(workspace.resolve())
                or stat.S_ISLNK(info.external_attr >> 16)):
            raise ValueError(f"Unexpected archive member: {info.filename}")
    archive.extractall(workspace)
project = workspace / 'cw1_gpu_smoke'
if not (project / 'scripts/part2/predict_depth.py').is_file():
    raise FileNotFoundError("The uploaded ZIP is not the expected smoke-test bundle")
cpu_reference = json.loads((project / 'cpu_reference.json').read_text())
print('Project:', project)
print('CPU reference seconds/frame:', cpu_reference['mean_inference_seconds'])

## 3. Install the smaller dependencies in an isolated environment

The environment inherits Colab's CUDA-enabled PyTorch/Torchvision. NumPy and the analysis dependencies are installed separately, so the notebook kernel and Colab's CUDA installation are preserved. Model inference and evaluation always run in fresh processes.

In [ ]:
env_root = Path('/content/cw1_depthpro_env')
subprocess.run([sys.executable, '-m', 'venv', '--without-pip', '--system-site-packages', str(env_root)], check=True)
python = env_root / 'bin/python'
subprocess.run([str(python), '-m', 'pip', 'install', '--quiet',
                'numpy==1.26.4', 'opencv-python-headless==4.10.0.84',
                'scipy', 'pandas', 'pyyaml', 'matplotlib', 'timm==1.0.30', 'pillow_heif'], check=True)
env_gpu_info = json.loads(subprocess.check_output([str(python), '-c', probe_code], text=True))
if not env_gpu_info['cuda_available'] or env_gpu_info['torch'] != gpu_info['torch']:
    raise RuntimeError("CUDA/PyTorch changed unexpectedly; stop and inspect the environment.")
print('CUDA PyTorch preserved:', env_gpu_info['torch'])

## 4. Install the pinned official model source

The source revision is fixed for reproducibility. `--no-deps` preserves the existing CUDA PyTorch installation; the smaller dependencies were installed above.

In [ ]:
source_root = Path('/content/cw1_depthpro_source')
source_commit = '9e65e4dbe9568d23c546fcec53302b10445e109e'
if not source_root.exists():
    subprocess.run(['git', 'clone', 'https://github.com/apple-aiml-research/ml-depth-pro.git', str(source_root)], check=True)
subprocess.run(['git', '-C', str(source_root), 'checkout', '--detach', source_commit], check=True)
subprocess.run([str(python), '-m', 'pip', 'install', '--quiet', '--no-deps', '-e', str(source_root)], check=True)
subprocess.run([str(python), '-c', 'import depth_pro, torch, numpy; print("Model imported; NumPy", numpy.__version__, "CUDA", torch.cuda.is_available())'], check=True)

## 5. Download and verify the checkpoint

This is the same official 1.9 GB checkpoint used for the laptop test. It is downloaded into the Colab VM; it is not uploaded from your laptop. Both size and SHA256 are checked before inference.

In [ ]:
import hashlib

checkpoint = Path('/content/cw1_depthpro_weights/depth_pro.pt')
checkpoint.parent.mkdir(parents=True, exist_ok=True)
expected_bytes = 1904446787
expected_sha256 = '3eb35ca68168ad3d14cb150f8947a4edf85589941661fdb2686259c80685c0ce'
if not checkpoint.is_file():
    partial = checkpoint.with_suffix('.partial')
    if not partial.is_file() or partial.stat().st_size != expected_bytes:
        subprocess.run(['curl', '-fL', '--retry', '3', '--continue-at', '-',
                        'https://ml-site.cdn-apple.com/models/depth-pro/depth_pro.pt',
                        '-o', str(partial)], check=True)
    candidate = partial
else:
    candidate = checkpoint
if candidate.stat().st_size != expected_bytes:
    raise ValueError(f"Incomplete checkpoint: {candidate.stat().st_size} bytes; rerun this cell to resume.")
digest = hashlib.sha256()
with candidate.open('rb') as handle:
    for block in iter(lambda: handle.read(8 * 1024**2), b''):
        digest.update(block)
if digest.hexdigest() != expected_sha256:
    raise ValueError("Checkpoint SHA256 mismatch; do not run inference with this file.")
if candidate != checkpoint:
    candidate.replace(checkpoint)
print('Verified checkpoint SHA256:', digest.hexdigest())

## 6. Check the three portable inputs

This inventory reads images and calibration without loading the network. It should report **3 RGB frames and 3 GT depth matches**, at 1200 ? 680.

In [ ]:
common_arguments = ['--sequence', 'replica_room0', '--max-frames', '3',
                    '--data-root', str(project / 'data'),
                    '--pyslam-root', str(project / 'calibration'),
                    '--checkpoint', str(checkpoint), '--device', 'cuda']
subprocess.run([str(python), str(project / 'scripts/part2/predict_depth.py'),
                *common_arguments, '--dry-run'], cwd=project, check=True)

## 7. Run the GPU smoke test

Run this cell once. It writes a prediction and timing record for each of the three frames. First-frame timing can include CUDA setup overhead. This initial test keeps float32 precision; memory fit on your assigned GPU is not yet verified.

If CUDA reports out-of-memory, stop the test and send the GPU details/error. A mixed-precision experiment should be measured separately rather than silently changing this comparison.

In [ ]:
import time

output_root = project / 'depth_maps'
started = time.perf_counter()
subprocess.run([str(python), str(project / 'scripts/part2/predict_depth.py'),
                *common_arguments, '--pred-root', str(output_root)], cwd=project, check=True)
whole_run_seconds = time.perf_counter() - started
manifest_path = output_root / 'replica_room0/manifest.json'
manifest = json.loads(manifest_path.read_text())
print('Whole process seconds (includes model startup):', round(whole_run_seconds, 2))

## 8. Compare the timings and record the hardware

The average of the last two frames reduces first-frame effects but remains a tiny sample. Extrapolations below assume this same rate for every image; they are planning estimates, not measured full-sequence runtimes. Script timing includes preprocessing and copying the result to CPU, and excludes file loading/saving.

In [ ]:
import statistics

entries = sorted(manifest['frames'].values(), key=lambda entry: entry['index'])
if len(entries) != 3:
    raise ValueError('Expected exactly three benchmark frames')
reference_entries = sorted(cpu_reference['frames'], key=lambda entry: entry['index'])
if cpu_reference['checkpoint_sha256'] != expected_sha256 or len(reference_entries) != len(entries):
    raise ValueError('GPU test and CPU reference do not use the same checkpoint/frame set')
for gpu_entry, cpu_entry in zip(entries, reference_entries):
    if gpu_entry['rgb_sha256'] != cpu_entry['rgb_sha256'] or gpu_entry['timestamp'] != cpu_entry['timestamp']:
        raise ValueError('GPU test inputs differ from the CPU reference')
times = [entry['inference_seconds'] for entry in entries]
mean_seconds = statistics.mean(times)
cpu_seconds = cpu_reference['mean_inference_seconds']
benchmark = {
    'kind': 'three_adjacent_frame_smoke_test', 'model': 'depth_pro',
    'source_commit': source_commit, 'checkpoint_sha256': expected_sha256,
    'gpu': env_gpu_info, 'precision': 'float32',
    'per_frame_inference_seconds': times, 'mean_inference_seconds': mean_seconds,
    'mean_last_two_seconds': statistics.mean(times[1:]),
    'cpu_reference_seconds': cpu_seconds, 'speedup_over_cpu': cpu_seconds / mean_seconds,
    'whole_prediction_process_seconds': whole_run_seconds,
    'estimated_replica_2000_hours': mean_seconds * 2000 / 3600,
    'estimated_all_5686_hours_if_same_rate': mean_seconds * 5686 / 3600,
}
results_dir = project / 'results'
results_dir.mkdir(exist_ok=True)
(results_dir / 'gpu_smoke_benchmark.json').write_text(json.dumps(benchmark, indent=2) + chr(10))
freeze = subprocess.check_output([str(python), '-m', 'pip', 'freeze'], text=True)
(results_dir / 'gpu_smoke_environment.txt').write_text(freeze)
print(json.dumps(benchmark, indent=2))

## 9. Evaluate raw and scale-adjusted depth, then view frame 0

GT depth is used only for scoring. Both predicted and GT maps use the same colour scale in the figure. This does not run or modify pySLAM.

In [ ]:
subprocess.run([str(python), str(project / 'scripts/part2/eval_depth.py'),
                '--sequence', 'replica_room0', '--max-frames', '3',
                '--data-root', str(project / 'data'), '--pred-root', str(output_root)], cwd=project, check=True)
subprocess.run([str(python), str(project / 'scripts/part2/visualize_depth.py'),
                '--sequence', 'replica_room0', '--frame-index', '0',
                '--data-root', str(project / 'data'), '--pred-root', str(output_root)], cwd=project,
                env={**__import__('os').environ, 'CW1_PYSLAM_ROOT': str(project / 'calibration')}, check=True)
from IPython.display import Image, display
display(Image(filename=str(project / 'results/figures/part2/replica_room0_frame000000.png')))

## 10. Download the results before ending the session

The ZIP includes predictions, their manifest, benchmark/hardware metadata, evaluation tables and the figure. It excludes model weights and input images. Browser download is explicit; no Google Drive mount or account-sharing step is required.

In [ ]:
archive_path = Path('/content/cw1_depthpro_gpu_smoke_results.zip')
with zipfile.ZipFile(archive_path, 'w', compression=zipfile.ZIP_DEFLATED) as archive:
    for folder in (project / 'depth_maps', project / 'results'):
        for file in sorted(folder.rglob('*')):
            if file.is_file():
                archive.write(file, file.relative_to(project))
print('Download:', archive_path.name, f'({archive_path.stat().st_size / 1024**2:.1f} MiB)')
files.download(str(archive_path))

## Decision after the test

- If the GPU time and depth quality are satisfactory, plan a separate full-data run and save results in chunks.
- Keep depth-generation hardware/time separate from later SLAM runtime with cached depth.
- Three nearby images do not establish accuracy or temporal stability for an entire sequence.
- Send back `gpu_smoke_benchmark.json` and the depth evaluation output before choosing the full run.

If no free GPU is available, the notebook deliberately stops. We can try later, continue the UCL support request, or benchmark a smaller model locally.